![image](https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)

# Use watsonx.ai Semantic Schema service to manage schema operations


#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.

## Notebook content

This notebook contains the steps and code demonstrating how to use the Semantic Schema service with Python SDK to create, improve, merge, and cluster schemas from documents.

Some familiarity with Python is helpful. This notebook uses Python 3.12.

## Learning goal

The purpose of this notebook is to demonstrate the usage of the Semantic Schema service and `ibm-watsonx-ai` Python SDK to manage schema operations including creating, improving, merging, and clustering schemas from documents.

## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Semantic Schema request preparation](#Semantic-Schema-request-preparation)
3. [Create Schemas](#Create-Schemas)
4. [Create data connections with source document](#Create-data-connections-with-source-document)
5. [Run Create Schemas job](#Run-Create-Schemas-job)
6. [Improve Schemas](#Improve-Schemas)
7. [Run Improve Schemas job](#Run-Improve-Schemas-job)
8. [Merge Schemas](#Merge-Schemas)
9. [Run Merge Schemas job](#Run-Merge-Schemas-job)
10. [Cluster Schemas](#Cluster-Schemas)
11. [Run Cluster Schemas job](#Run-Cluster-Schemas-job)
12. [Summary and next steps](#Summary-and-next-steps)


<a id="Set-up-the-environment"></a>

## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

- Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).


### Install dependencies

**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [ ]:
%pip install wget | tail -n 1
%pip install "ibm-watsonx-ai>=1.5.9" | tail -n 1

### Defining the watsonx.ai credentials

This cell defines the watsonx.ai credentials required to work with watsonx Semantic Schema service.

**Action:** Provide the IBM Cloud user API key. For details, see the <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">documentation</a>.

**Note:** For AWS environments, follow this <a href="https://cloud.ibm.com/docs/account?topic=account-service_credentials&interface=ui" target="_blank" rel="noopener no referrer">documentation page</a> to obtain your API key.

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

### Defining the project ID

The Semantic Schema service requires a deployment context provided by a project or space. In this notebook, we use the space context and obtain the identifier from the environment when available. Otherwise, provide the space ID manually.


In [2]:
import os

try:
    space_id = os.environ["SPACE_ID"]
except KeyError:
    space_id = input("Please enter your space_id (hit enter): ")

### APIClient initialization


In [3]:
from ibm_watsonx_ai import APIClient

api_client = APIClient(credentials=credentials, space_id=space_id)

<a id="Semantic-Schema-request-preparation"></a>

## Semantic Schema request preparation

After the service credentials and deployment context are configured, you can initialize the Semantic Schema manager. In this notebook, the `SemanticSchema` class is used to submit and manage create, improve, merge, and cluster jobs.

In [4]:
from ibm_watsonx_ai.foundation_models.semantic_schema import SemanticSchema

semantic_schema = SemanticSchema(api_client=api_client)

### Available Operation Handlers

The `SemanticSchema` object provides handlers for different schema operations:

- `create` – for creating new schemas
- `improve` – for refining existing schemas
- `merge` – for combining multiple schemas
- `cluster` – for grouping similar schema structures

### Semantic Schema Parameters

When running a job, the parameters for the semantic schema pipeline can be specified. For more details about available parameters see api-docs references:

- [Create Schema](https://cloud.ibm.com/apidocs/watsonx-ai#create-schema)
- [Improve Schema](https://cloud.ibm.com/apidocs/watsonx-ai#improve-schema)
- [Merge Schema](https://cloud.ibm.com/apidocs/watsonx-ai#merge-schema)
- [Cluster Schema](https://cloud.ibm.com/apidocs/watsonx-ai#cluster-schema)


### Helper Functions

Two utility functions used across all schema operations: one to poll job status until completion, and one to display component info.


In [5]:
import inspect
import time

RED = "\033[31m"
RESET = "\033[0m"


def wait_for_job_completion(component, job_id: str, poll_interval: int = 5):
    """Wait until the semantic schema job completes or fails.

    Args:
        component: The semantic schema component (create, improve, merge, or cluster)
        job_id: The job ID to monitor
        poll_interval: Time in seconds between status checks (default: 5)
    """
    while True:
        status = component.get_status(job_id)

        if status == "completed" or "fail" in status.lower():
            print(f"\n{status}")
            break

        print(".", end="", flush=True)
        time.sleep(poll_interval)


def show_info(obj):
    print(f"{RED}Type:{RESET}        {type(obj).__name__}")
    print(f"{RED}String form:{RESET} {obj}")
    print(f"{RED}Docstring:{RESET}\n{inspect.getdoc(obj)}")

<a id="Create-Schemas"></a>

## Create Schemas

In this section, you create a semantic schema from a source document stored in the current space. The generated schema can then be reviewed and used as input to later improvement, merge, or clustering workflows.

In [6]:
show_info(semantic_schema.create)

Type:        CreateSchemas
String form: <ibm_watsonx_ai.foundation_models.semantic_schema.create_schemas.CreateSchemas object at 0x117cf7c20>
Docstring:
Handle schema creation operations.

This class provides methods to create new schemas from documents through job-based
operations. Schema creation analyzes document structure and generates appropriate
schema definitions automatically.


<a id="Create-data-connections-with-source-document"></a>

## Create data connections with source document

In this section, we will download a sample PDF document and create a data connection to it. This document will be used as input for the semantic schema creation process.

### Download source document

We will download the `IBM_wikipedia.pdf` file which contains information about IBM that will be used to create a semantic schema.

In [7]:
import os

import wget

filename = "IBM_wikipedia.pdf"
url = "https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/data/foundation_models/IBM_wikipedia.pdf"

if not os.path.isfile(filename):
    wget.download(url, out=filename)

### Create semantic schema document reference

Create a `DataConnection` object that points to the uploaded PDF in the current space. This object is passed to the create schemas job as the input document reference.

In [8]:
from ibm_watsonx_ai.helpers import ContainerLocation, DataConnection

data_reference = DataConnection(
    location=ContainerLocation(path=filename),
)
data_reference.set_client(api_client)
data_reference.write(
    data=filename,
)
data_reference

{'type': 'container', 'location': {'path': 'IBM_wikipedia.pdf'}}

### Define parameters for the Create Schemas job

The parameters below configure how the service analyzes the source document. In this example, high-quality extraction is enabled together with OCR so that scanned or image-based text can also be processed.

In [9]:
from ibm_watsonx_ai.foundation_models.schema import (
    CreateSchemasMode,
    CreateSchemasParameters,
    OCRMode,
)

create_parameters = CreateSchemasParameters(
    mode=CreateSchemasMode.HIGH_QUALITY,
    ocr_mode=OCRMode.ENABLED,
    enable_grounding=False,
    languages=["en"],
)

<a id="Run-Create-Schemas-job"></a>

## Run Create Schemas job

In this section, we will run a semantic schema creation job using the document reference and parameters we defined earlier. The job will analyze the document and generate a semantic schema.


In [10]:
create_schemas_job_details = semantic_schema.create.run_job(
    document_reference=data_reference,
    parameters=create_parameters,
)
create_schemas_job_details

{'metadata': {'id': '35e68406-f11a-44a5-b208-f1778569ebcb',
  'created_at': '2026-04-22T12:55:23.810Z',
  'space_id': 'ab88d81c-e1dd-41ea-9dd7-ff98931c4427'},
 'entity': {'document_reference': {'type': 'container',
   'location': {'path': 'IBM_wikipedia.pdf'}},
  'parameters': {'mode': 'high_quality',
   'ocr_mode': 'enabled',
   'languages': ['en'],
   'enable_grounding': False,
   'max_pages_to_process': 20},
  'results': {'status': 'submitted', 'schema': {}}},
 'system': {}}

In [11]:
create_schemas_job_id = semantic_schema.create.get_job_id(create_schemas_job_details)
create_schemas_job_id

'35e68406-f11a-44a5-b208-f1778569ebcb'

To wait until the create schemas job completes, run the following cell:


In [12]:
wait_for_job_completion(semantic_schema.create, create_schemas_job_id)

..
completed


We can list create schemas jobs using the list method.


In [13]:
semantic_schema.create.list_jobs()

,CREATE_SCHEMA_JOB_ID,CREATED,STATUS
0,35e68406-f11a-44a5-b208-f1778569ebcb,2026-04-22T12:55:23.810Z,completed


To get details of a particular create schemas request, run the following:


In [14]:
semantic_schema.create.get_job_details(create_schemas_job_id)

{'entity': {'document_reference': {'location': {'path': 'IBM_wikipedia.pdf'},
   'type': 'container'},
  'parameters': {'enable_grounding': False,
   'languages': ['en'],
   'max_pages_to_process': 20,
   'mode': 'high_quality',
   'ocr_mode': 'enabled'},
  'results': {'completed_at': '2026-04-22T12:55:33.238Z',
   'number_pages_processed': 1,
   'running_at': '2026-04-22T12:55:26.397Z',
   'schema': {'document_description': "A document providing an overview of a multinational technology company, specifically IBM. Key fields include company name, business name, headquarters location, global presence, stock market status, research facilities, and notable achievements such as patent records. The document highlights IBM's extensive research capabilities and its significant contributions to innovation.",
    'document_type': 'Company_Profile'},
   'status': 'completed',
   'total_pages': 1}},
 'metadata': {'created_at': '2026-04-22T12:55:23.810Z',
  'id': '35e68406-f11a-44a5-b208-f1778569e

### Results examination for create schemas job

Once the job is completed, we can use the `get_results` method to check results.


In [15]:
create_schemas_results = semantic_schema.create.get_results(create_schemas_job_id)
create_schemas_results

{'completed_at': '2026-04-22T12:55:33.238Z',
 'number_pages_processed': 1,
 'running_at': '2026-04-22T12:55:26.397Z',
 'schema': {'document_description': "A document providing an overview of a multinational technology company, specifically IBM. Key fields include company name, business name, headquarters location, global presence, stock market status, research facilities, and notable achievements such as patent records. The document highlights IBM's extensive research capabilities and its significant contributions to innovation.",
  'document_type': 'Company_Profile'},
 'status': 'completed',
 'total_pages': 1}

In [16]:
for key, value in create_schemas_results["schema"].items():
    print(f"{key}: {value}")

document_description: A document providing an overview of a multinational technology company, specifically IBM. Key fields include company name, business name, headquarters location, global presence, stock market status, research facilities, and notable achievements such as patent records. The document highlights IBM's extensive research capabilities and its significant contributions to innovation.
document_type: Company_Profile


Furthermore, to delete Create Schemas job run use `delete_job()` method.

In [17]:
semantic_schema.create.delete_job(create_schemas_job_id)

'SUCCESS'

<a id="Improve-Schemas"></a>

## Improve Schemas

In [18]:
show_info(semantic_schema.improve)

Type:        ImproveSchemas
String form: <ibm_watsonx_ai.foundation_models.semantic_schema.improve_schemas.ImproveSchemas object at 0x112389bb0>
Docstring:
Handle schema improvement operations.

This class provides methods to improve existing schemas through job-based operations.
Schema improvement can include refining field definitions, adding metadata, optimizing
structure, and enhancing schema quality based on data analysis.


### Define parameters for the Improve Schemas job

The improve operation enriches an existing schema definition by expanding its description and inferred structure. Provide a minimal schema draft, and the service returns a more complete semantic representation.

In [19]:
from ibm_watsonx_ai.foundation_models.schema import (
    ImproveSchemasFields,
    ImproveSchemasParameters,
    ImproveSchemasSchemaDefinition,
)

improve_parameters = ImproveSchemasParameters(
    schema=ImproveSchemasSchemaDefinition(
        document_type="Passport",
        document_description="Passport document to get the schema",
        fields={
            "name": ImproveSchemasFields(
                description="Full name of the passport holder", example="John Doe"
            ),
            "passport_number": ImproveSchemasFields(
                description="Unique passport identifier number", example="AB1234567"
            ),
        },
    )
)

<a id="Run-Improve-Schemas-job"></a>

## Run Improve Schemas job

In this section, we will run a semantic schema improve job using the parameters we defined earlier.


In [20]:
improve_schemas_job_details = semantic_schema.improve.run_job(
    parameters=improve_parameters
)

improve_schemas_job_details

{'metadata': {'id': '352394a9-3444-4412-9101-ac31fff2f369',
  'created_at': '2026-04-22T12:55:54.916Z',
  'space_id': 'ab88d81c-e1dd-41ea-9dd7-ff98931c4427'},
 'entity': {'parameters': {'schema': {'document_type': 'Passport',
    'document_description': 'Passport document to get the schema',
    'fields': {'name': {'description': 'Full name of the passport holder',
      'example': 'John Doe'},
     'passport_number': {'description': 'Unique passport identifier number',
      'example': 'AB1234567'}}}},
  'results': {'status': 'submitted', 'schemas': {}}},
 'system': {}}

In [21]:
improve_schemas_job_id = semantic_schema.improve.get_job_id(improve_schemas_job_details)
improve_schemas_job_id

'352394a9-3444-4412-9101-ac31fff2f369'

To wait until the improve schemas job completes, run the following cell:


In [22]:
wait_for_job_completion(semantic_schema.improve, improve_schemas_job_id)

.
completed


We can list improve schemas jobs using the list method.


In [25]:
semantic_schema.improve.list_jobs()

,IMPROVE_SCHEMA_JOB_ID,CREATED,STATUS
0,352394a9-3444-4412-9101-ac31fff2f369,2026-04-22T12:55:54.916Z,completed


To get details of a particular improve schemas request, run the following:


In [26]:
semantic_schema.improve.get_job_details(improve_schemas_job_id)

{'entity': {'parameters': {'schema': {'document_description': 'Passport document to get the schema',
    'document_type': 'Passport',
    'fields': {'name': {'description': 'Full name of the passport holder',
      'example': 'John Doe'},
     'passport_number': {'description': 'Unique passport identifier number',
      'example': 'AB1234567'}}}},
  'results': {'completed_at': '2026-04-22T12:55:59.196Z',
   'running_at': '2026-04-22T12:55:55.705Z',
   'schema': {'document_description': 'A Passport is an official travel document that verifies the identity and nationality of the holder, facilitating international travel. It is issued by a government to its citizens and is used for border control and identification purposes. Key fields include: name, passport_number',
    'document_type': 'Passport',
    'fields': {'name': {'description': 'Full name of the passport holder',
      'example': 'John Doe'},
     'passport_number': {'description': 'Unique passport identifier number',
      'ex

### Results examination for improve schemas job

Once the job is completed, we can use the `get_results` method to check results.


In [27]:
improve_schemas_results = semantic_schema.improve.get_results(improve_schemas_job_id)
improve_schemas_results

{'completed_at': '2026-04-22T12:55:59.196Z',
 'running_at': '2026-04-22T12:55:55.705Z',
 'schema': {'document_description': 'A Passport is an official travel document that verifies the identity and nationality of the holder, facilitating international travel. It is issued by a government to its citizens and is used for border control and identification purposes. Key fields include: name, passport_number',
  'document_type': 'Passport',
  'fields': {'name': {'description': 'Full name of the passport holder',
    'example': 'John Doe'},
   'passport_number': {'description': 'Unique passport identifier number',
    'example': 'AB1234567'}}},
 'status': 'completed'}

In [28]:
for key, value in improve_schemas_results["schema"].items():
    print(f"{key}: {value}")

document_description: A Passport is an official travel document that verifies the identity and nationality of the holder, facilitating international travel. It is issued by a government to its citizens and is used for border control and identification purposes. Key fields include: name, passport_number
document_type: Passport
fields: {'name': {'description': 'Full name of the passport holder', 'example': 'John Doe'}, 'passport_number': {'description': 'Unique passport identifier number', 'example': 'AB1234567'}}


Furthermore, to delete Improve Schemas job run use `delete_job()` method.

In [29]:
semantic_schema.improve.delete_job(improve_schemas_job_id)

'SUCCESS'

<a id="Merge-Schemas"></a>

## Merge Schemas

In [30]:
show_info(semantic_schema.merge)

Type:        MergeSchemas
String form: <ibm_watsonx_ai.foundation_models.semantic_schema.merge_schemas.MergeSchemas object at 0x114f29f70>
Docstring:
Handle schema merging operations.

This class provides methods to merge multiple schemas into a unified schema through
job-based operations. Schema merging combines fields from multiple source schemas,
resolves conflicts, and creates a consolidated schema structure.


### Define parameters for the Merge Schemas job

The merge operation combines multiple schema definitions into a single consolidated schema. This is useful when related document types share common semantics and should be represented through one unified structure.

In [31]:
from ibm_watsonx_ai.foundation_models.schema import (
    MergeSchemasFields,
    MergeSchemasParameters,
    MergeSchemasSchemaDefinition,
)

merge_schemas_parameters = MergeSchemasParameters(
    schemas=[
        MergeSchemasSchemaDefinition(
            document_type="Passport",
            document_description="Passport document to get the schema",
            fields={
                "name": MergeSchemasFields(
                    description="Full name of the passport holder", example="John Doe"
                ),
                "passport_number": MergeSchemasFields(
                    description="Unique passport identifier number", example="AB1234567"
                ),
            },
        ),
        MergeSchemasSchemaDefinition(
            document_type="National ID Card",
            document_description="National ID Cards are government-issued identification documents",
            fields={
                "name": MergeSchemasFields(
                    description="Full name of the ID card holder", example="Jane Doe"
                ),
                "id_number": MergeSchemasFields(
                    description="Unique national ID card number", example="ID-987654321"
                ),
            },
        ),
    ]
)

<a id="Run-Merge-Schemas-job"></a>

## Run Merge Schemas job

In this section, we will run a semantic schema merge job to combine multiple schemas into a unified schema.


In [32]:
merge_schemas_job_details = semantic_schema.merge.run_job(
    parameters=merge_schemas_parameters
)

merge_schemas_job_details

{'metadata': {'id': '36e8de92-b5ca-44e7-9121-1bfd2000424a',
  'created_at': '2026-04-22T12:56:49.782Z',
  'space_id': 'ab88d81c-e1dd-41ea-9dd7-ff98931c4427'},
 'entity': {'parameters': {'schemas': [{'document_type': 'Passport',
     'document_description': 'Passport document to get the schema',
     'fields': {'name': {'description': 'Full name of the passport holder',
       'example': 'John Doe'},
      'passport_number': {'description': 'Unique passport identifier number',
       'example': 'AB1234567'}}},
    {'document_type': 'National ID Card',
     'document_description': 'National ID Cards are government-issued identification documents',
     'fields': {'id_number': {'description': 'Unique national ID card number',
       'example': 'ID-987654321'},
      'name': {'description': 'Full name of the ID card holder',
       'example': 'Jane Doe'}}}]},
  'results': {'status': 'submitted', 'schemas': {}}},
 'system': {}}

In [33]:
merge_schemas_job_id = semantic_schema.merge.get_job_id(merge_schemas_job_details)
merge_schemas_job_id

'36e8de92-b5ca-44e7-9121-1bfd2000424a'

To wait until the merge schemas job completes, run the following cell:


In [34]:
wait_for_job_completion(semantic_schema.merge, merge_schemas_job_id)

.
completed


We can list merge schemas jobs using the list method.


In [35]:
semantic_schema.merge.list_jobs()

,MERGE_SCHEMA_JOB_ID,CREATED,STATUS
0,36e8de92-b5ca-44e7-9121-1bfd2000424a,2026-04-22T12:56:49.782Z,completed


To get details of a particular merge schemas request, run the following:


In [36]:
semantic_schema.merge.get_job_details(merge_schemas_job_id)

{'entity': {'parameters': {'schemas': [{'document_description': 'Passport document to get the schema',
     'document_type': 'Passport',
     'fields': {'name': {'description': 'Full name of the passport holder',
       'example': 'John Doe'},
      'passport_number': {'description': 'Unique passport identifier number',
       'example': 'AB1234567'}}},
    {'document_description': 'National ID Cards are government-issued identification documents',
     'document_type': 'National ID Card',
     'fields': {'id_number': {'description': 'Unique national ID card number',
       'example': 'ID-987654321'},
      'name': {'description': 'Full name of the ID card holder',
       'example': 'Jane Doe'}}}]},
  'results': {'completed_at': '2026-04-22T12:56:54.802Z',
   'running_at': '2026-04-22T12:56:50.102Z',
   'schema': {'document_description': 'Identification documents including Passports and National ID Cards, which are government-issued identification documents',
    'document_type': 'Iden

### Results examination for merge schemas job

Once the job is completed, we can use the `get_results` method to check results.


In [37]:
merge_schemas_results = semantic_schema.merge.get_results(merge_schemas_job_id)
merge_schemas_results

{'completed_at': '2026-04-22T12:56:54.802Z',
 'running_at': '2026-04-22T12:56:50.102Z',
 'schema': {'document_description': 'Identification documents including Passports and National ID Cards, which are government-issued identification documents',
  'document_type': 'Identification Document',
  'fields': {'document_number': {'description': 'Unique identifier number for the identification document',
    'example': 'AB1234567'},
   'name': {'description': 'Full name of the document holder',
    'example': 'John Doe'}}},
 'status': 'completed'}

In [38]:
for key, value in merge_schemas_results["schema"].items():
    print(f"{key}: {value}")

document_description: Identification documents including Passports and National ID Cards, which are government-issued identification documents
document_type: Identification Document
fields: {'document_number': {'description': 'Unique identifier number for the identification document', 'example': 'AB1234567'}, 'name': {'description': 'Full name of the document holder', 'example': 'John Doe'}}


Furthermore, to delete Merge Schemas job run use `delete_job()` method.

In [39]:
semantic_schema.merge.delete_job(merge_schemas_job_id)

'SUCCESS'

<a id="Cluster-Schemas"></a>

## Cluster Schemas

In [40]:
show_info(semantic_schema.cluster)

Type:        ClusterSchemas
String form: <ibm_watsonx_ai.foundation_models.semantic_schema.cluster_schemas.ClusterSchemas object at 0x10ca9e8d0>
Docstring:
Handle schema clustering operations.

This class provides methods to cluster and group schemas based on similarity through
job-based operations. Schema clustering analyzes multiple schemas to identify patterns,
group similar schemas together, and discover schema relationships.


### Define parameters for the Cluster Schemas job

The cluster operation groups related schema definitions based on similarity. Each input item contains a document name together with its schema so the output can preserve the relationship between clustered schemas and their source labels.

In [41]:
from ibm_watsonx_ai.foundation_models.schema import (
    ClusterSchemasDocument,
    ClusterSchemasFields,
    ClusterSchemasParameters,
    ClusterSchemasSchemaDefinition,
)

cluster_schemas_parameters = ClusterSchemasParameters(
    schemas=[
        ClusterSchemasDocument(
            document_name="Passport",
            schema=ClusterSchemasSchemaDefinition(
                document_type="Passport",
                document_description="Passport document to get the schema",
                fields={
                    "name": ClusterSchemasFields(
                        description="Full name of the passport holder",
                        example="John Doe",
                    ),
                    "passport_number": ClusterSchemasFields(
                        description="Unique passport identifier number",
                        example="AB1234567",
                    ),
                },
            ),
        ),
        ClusterSchemasDocument(
            document_name="National_ID_Card",
            schema=ClusterSchemasSchemaDefinition(
                document_type="National ID Card",
                document_description="National ID Cards are government-issued identification documents",
                fields={
                    "name": ClusterSchemasFields(
                        description="Full name of the ID card holder",
                        example="Jane Doe",
                    ),
                    "id_number": ClusterSchemasFields(
                        description="Unique national ID card number",
                        example="ID-987654321",
                    ),
                },
            ),
        ),
    ]
)

<a id="Run-Cluster-Schemas-job"></a>

## Run Cluster Schemas job

In this section, we will run a semantic schema cluster job to group similar schemas together.


In [42]:
cluster_schemas_job_details = semantic_schema.cluster.run_job(
    parameters=cluster_schemas_parameters
)

cluster_schemas_job_details

{'metadata': {'id': 'e9fffee6-c935-4087-9773-23dbbb5269e7',
  'created_at': '2026-04-22T12:57:08.945Z',
  'space_id': 'ab88d81c-e1dd-41ea-9dd7-ff98931c4427'},
 'entity': {'parameters': {'schemas': [{'document_name': 'Passport',
     'schema': {'document_type': 'Passport',
      'document_description': 'Passport document to get the schema',
      'fields': {'name': {'description': 'Full name of the passport holder',
        'example': 'John Doe'},
       'passport_number': {'description': 'Unique passport identifier number',
        'example': 'AB1234567'}}}},
    {'document_name': 'National_ID_Card',
     'schema': {'document_type': 'National ID Card',
      'document_description': 'National ID Cards are government-issued identification documents',
      'fields': {'id_number': {'description': 'Unique national ID card number',
        'example': 'ID-987654321'},
       'name': {'description': 'Full name of the ID card holder',
        'example': 'Jane Doe'}}}}]},
  'results': {'status'

In [43]:
cluster_schemas_job_id = semantic_schema.cluster.get_job_id(cluster_schemas_job_details)
cluster_schemas_job_id

'e9fffee6-c935-4087-9773-23dbbb5269e7'

To wait until the cluster schemas job completes, run the following cell:


In [44]:
wait_for_job_completion(semantic_schema.cluster, cluster_schemas_job_id)

.
completed


We can list cluster schemas jobs using the list method.


In [45]:
semantic_schema.cluster.list_jobs()

,CLUSTER_SCHEMA_JOB_ID,CREATED,STATUS
0,e9fffee6-c935-4087-9773-23dbbb5269e7,2026-04-22T12:57:08.945Z,completed


To get details of a particular cluster schemas request, run the following:


In [46]:
semantic_schema.cluster.get_job_details(cluster_schemas_job_id)

{'entity': {'parameters': {'schemas': [{'document_name': 'Passport',
     'schema': {'document_description': 'Passport document to get the schema',
      'document_type': 'Passport',
      'fields': {'name': {'description': 'Full name of the passport holder',
        'example': 'John Doe'},
       'passport_number': {'description': 'Unique passport identifier number',
        'example': 'AB1234567'}}}},
    {'document_name': 'National_ID_Card',
     'schema': {'document_description': 'National ID Cards are government-issued identification documents',
      'document_type': 'National ID Card',
      'fields': {'id_number': {'description': 'Unique national ID card number',
        'example': 'ID-987654321'},
       'name': {'description': 'Full name of the ID card holder',
        'example': 'Jane Doe'}}}}]},
  'results': {'completed_at': '2026-04-22T12:57:12.628Z',
   'running_at': '2026-04-22T12:57:09.673Z',
   'schemas': [[{'document_name': 'Passport',
      'schema': {'document_descr

### Results examination for cluster schemas job

Once the job is completed, we can use the `get_results` method to check results.


In [47]:
cluster_schemas_results = semantic_schema.cluster.get_results(cluster_schemas_job_id)
cluster_schemas_results

{'completed_at': '2026-04-22T12:57:12.628Z',
 'running_at': '2026-04-22T12:57:09.673Z',
 'schemas': [[{'document_name': 'Passport',
    'schema': {'document_description': 'Passport document to get the schema',
     'document_type': 'Passport',
     'fields': {'name': {'description': 'Full name of the passport holder',
       'example': 'John Doe'},
      'passport_number': {'description': 'Unique passport identifier number',
       'example': 'AB1234567'}}}}],
  [{'document_name': 'National_ID_Card',
    'schema': {'document_description': 'National ID Cards are government-issued identification documents',
     'document_type': 'National ID Card',
     'fields': {'id_number': {'description': 'Unique national ID card number',
       'example': 'ID-987654321'},
      'name': {'description': 'Full name of the ID card holder',
       'example': 'Jane Doe'}}}}]],
 'status': 'completed'}

In [48]:
for index, schema_group in enumerate(
    cluster_schemas_results.get("schemas", []), start=1
):
    print(f"Schema group {index}: {schema_group}")

Schema group 1: [{'document_name': 'Passport', 'schema': {'document_description': 'Passport document to get the schema', 'document_type': 'Passport', 'fields': {'name': {'description': 'Full name of the passport holder', 'example': 'John Doe'}, 'passport_number': {'description': 'Unique passport identifier number', 'example': 'AB1234567'}}}}]
Schema group 2: [{'document_name': 'National_ID_Card', 'schema': {'document_description': 'National ID Cards are government-issued identification documents', 'document_type': 'National ID Card', 'fields': {'id_number': {'description': 'Unique national ID card number', 'example': 'ID-987654321'}, 'name': {'description': 'Full name of the ID card holder', 'example': 'Jane Doe'}}}}]


Furthermore, to delete Cluster Schemas job run use `delete_job()` method.

In [49]:
semantic_schema.cluster.delete_job(cluster_schemas_job_id)

'SUCCESS'

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to use the `SemanticSchema` manager to:
- Initialize the Semantic Schema service
- Create data connections for source documents
- Run schema creation jobs with custom parameters
- Improve existing schemas to enhance their quality and completeness
- Merge multiple schemas into a unified schema
- Cluster similar schemas together for better organization
- Monitor job status and retrieve results
- Manage and delete schema jobs

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts.

### Authors and Maintainers

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2026 IBM. This notebook and its source code are released under the terms of the MIT License.